# 13 · Floating point accuracy and numerical stability

**Level:** Advanced · **Practice time:** 30–50 minutes

[← Previous](12_vectorization_and_performance.ipynb) · [Course guide](../README.md) · [Next →](14_advanced_array_patterns.ipynb)

**You will learn:** Use tolerances, avoid overflow and cancellation, and inspect precision limits.

**Prerequisites:** Notebooks 00–12.

## How to learn with this notebook

1. Run cells from top to bottom. Predict results before executing each example.
2. Do the **Try it** experiment by editing the example or adding a cell below it.
3. Replace `None` in each exercise with your answer, then rerun its check cell.
4. Consult the solutions at the end only after trying. An untouched exercise prints **Pending**, so Run All succeeds without claiming you solved it.
5. Complete the independent challenge, then restart the kernel and run all cells again.

Each notebook imports its own dependencies and creates its own data. You do not need variables from another notebook. The supplied outputs show a verified reference run; rerun cells to explore your changes.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
print("NumPy version:", np.__version__)

NumPy version: 2.5.3


## Step 1 · Approximate arithmetic needs meaningful tolerances

Many decimal fractions have no exact binary representation. `isclose` and `assert_allclose` compare with absolute and relative tolerances. Choose tolerances from the scale and accuracy needs of your problem; default tolerances are not a universal specification. Near zero, absolute tolerance matters especially.

In [2]:
value = 0.1 + 0.2
print("Stored sum:", repr(value))
print("Exact equality:", value == 0.3)
print("Close:", np.isclose(value, 0.3, rtol=1e-12, atol=1e-15))
print("float32 epsilon:", np.finfo(np.float32).eps)
print("float64 epsilon:", np.finfo(np.float64).eps)
np.testing.assert_allclose(value, 0.3, rtol=1e-12, atol=1e-15)

Stored sum: 0.30000000000000004
Exact equality: False
Close: True
float32 epsilon: 1.1920929e-07
float64 epsilon: 2.220446049250313e-16


**Try it:** Compare 1e-12 and 0 using different absolute tolerances.

## Step 2 · Preserve small changes and avoid overflow

Subtracting nearly equal quantities can lose meaningful digits. Use `log1p(x)` for log(1+x) and `expm1(x)` for exp(x)-1 when x is small. NumPy integer arithmetic has fixed-width limits, so promote the dtype before arithmetic that may overflow.

In [3]:
tiny = 1e-16
print("Naive log:", np.log(1 + tiny))
print("log1p:", np.log1p(tiny))
print("Naive exp change:", np.exp(tiny) - 1)
print("expm1:", np.expm1(tiny))
small = np.array([120], dtype=np.int8)
print("Overflowed int8 sum:", small + small)
print("Promoted sum:", small.astype(np.int64) + small.astype(np.int64))
np.testing.assert_allclose(np.log1p(tiny), tiny, rtol=1e-12, atol=0)

Naive log: 0.0
log1p: 1e-16
Naive exp change: 0.0
expm1: 1e-16
Overflowed int8 sum: [-16]
Promoted sum: [240]


**Try it:** Find a value for which float32 rounds 1 + x back to 1, while float64 does not.

## Step 3 · Stable softmax

Softmax converts finite scores into positive weights summing to one. Direct exponentials can overflow for large scores. Subtract the row maximum first: the largest exponent becomes exp(0)=1, and the normalized result is mathematically unchanged. Very small probabilities can still underflow to zero.

In [4]:
def softmax(scores):
    scores = np.asarray(scores, dtype=np.float64)
    shifted = scores - scores.max(axis=-1, keepdims=True)
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum(axis=-1, keepdims=True)

scores = np.array([[1000., 1001., 1002.], [-1000., -1000., -999.]])
probabilities = softmax(scores)
print(probabilities)
assert np.isfinite(probabilities).all()
np.testing.assert_allclose(probabilities.sum(axis=1), 1)
np.testing.assert_allclose(softmax(scores + 500), probabilities)

[[0.09   0.2447 0.6652]
 [0.2119 0.2119 0.5761]]


**Try it:** Add a test for equal scores and another for a row containing only one score.

## Your practice exercises

Use the named answer variables. The checks verify your result; they do not require the same approach as the reference solution. If an assertion fails, read the expected value and inspect your shape and dtype.

### Exercise 1

Compute log(1 + x) accurately for very small x.

In [5]:
tiny_values = np.array([1e-16, 1e-12, 1e-8])

accurate_logs = None  # TODO: replace None with your answer

In [6]:
if accurate_logs is None:
    print("Pending: complete Exercise 1 first.")
else:
    np.testing.assert_allclose(accurate_logs, np.log1p(tiny_values), rtol=1e-12, atol=0)
    print("Exercise 1 passed!")

Pending: complete Exercise 1 first.


### Exercise 2

Use softmax to convert [10000, 10000] into normalized probabilities.

In [7]:
large_scores = np.array([10000., 10000.])

weights = None  # TODO: replace None with your answer

In [8]:
if weights is None:
    print("Pending: complete Exercise 2 first.")
else:
    assert np.isfinite(weights).all()
    np.testing.assert_allclose(weights, [0.5, 0.5])
    print("Exercise 2 passed!")

Pending: complete Exercise 2 first.


## Independent challenge

Compare summing float32 values with default accumulation and dtype=float64. Construct a dataset with very different magnitudes and explain any difference.

Add code and explanation cells below. Decide on expected results before checking your implementation.

In [9]:
# Your independent challenge work goes here.

## Before you move on

- Explain the main idea in your own words without reading the examples.
- Make both exercise checks print **passed**.
- Try at least one changed input or edge case and explain the result.
- Finish the independent challenge or write down a specific question to revisit.

Expertise comes from repeated practice and debugging unfamiliar problems. Completing these notebooks gives you a broad practical foundation, not a substitute for that experience.

---

## Reference solutions — try first

These cells use separate reference variables, so they do not fill in your exercise answers. They also check the reference result. Compare your reasoning as well as the output.

### Solution 1

Compute log(1 + x) accurately for very small x.

**Why it works:** log1p computes the small logarithmic change without first rounding 1+x to a nearby representable value.

In [10]:
reference_accurate_logs = np.log1p(tiny_values)
np.testing.assert_allclose(reference_accurate_logs, np.log1p(tiny_values), rtol=1e-12, atol=0)
print(reference_accurate_logs)

[0. 0. 0.]


### Solution 2

Use softmax to convert [10000, 10000] into normalized probabilities.

**Why it works:** Subtracting the maximum makes both shifted scores zero. Their exponentials are both one and normalize to equal weights.

In [11]:
reference_weights = softmax(large_scores)
assert np.isfinite(reference_weights).all()
np.testing.assert_allclose(reference_weights, [0.5, 0.5])
print(reference_weights)

[0.5 0.5]


## Continue learning

- [Official NumPy documentation: numpy.allclose ](https://numpy.org/doc/stable/reference/generated/numpy.allclose.html)
- [Official NumPy documentation: numpy.log1p ](https://numpy.org/doc/stable/reference/generated/numpy.log1p.html)

[Next →](14_advanced_array_patterns.ipynb)